In [2]:
import pandas as pd 
import numpy as np
from sklearn.model_selection import train_test_split


LOAD AND INSPECT

In [3]:
df = pd.read_csv("uae_properties.csv")

df.shape
df.head()

,area,bedrooms,size_sqft,age_years,freehold,price_aed
0,Dubai Marina,0,562.0,13,1,664000
1,Mirdif,3,1961.0,8,0,1343000
2,Arabian Ranches,4,2670.0,13,1,3413000
3,JVC,4,2593.0,4,0,2557000
4,JVC,1,630.0,2,1,618000


INSPECT

In [4]:
df.info()
df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   area       500 non-null    str    
 1   bedrooms   500 non-null    int64  
 2   size_sqft  482 non-null    float64
 3   age_years  500 non-null    int64  
 4   freehold   500 non-null    int64  
 5   price_aed  500 non-null    int64  
dtypes: float64(1), int64(4), str(1)
memory usage: 23.6 KB


,bedrooms,size_sqft,age_years,freehold,price_aed
count,500.000000,482.000000,500.000000,500.000000,5.000000e+02
mean,2.036000,1399.087137,11.714000,0.670000,1.533224e+06
std,1.433462,811.476161,7.137092,0.470684,1.047626e+06
min,0.000000,350.000000,0.000000,0.000000,2.050000e+05
25%,1.000000,558.500000,6.000000,0.000000,6.177500e+05
50%,2.000000,1331.000000,11.000000,1.000000,1.319500e+06
75%,3.000000,2033.500000,18.000000,1.000000,2.208250e+06
max,4.000000,2840.000000,24.000000,1.000000,5.193000e+06


In [5]:
df.describe().round(0)

,bedrooms,size_sqft,age_years,freehold,price_aed
count,500.0,482.0,500.0,500.0,500.0
mean,2.0,1399.0,12.0,1.0,1533224.0
std,1.0,811.0,7.0,0.0,1047626.0
min,0.0,350.0,0.0,0.0,205000.0
25%,1.0,558.0,6.0,0.0,617750.0
50%,2.0,1331.0,11.0,1.0,1319500.0
75%,3.0,2034.0,18.0,1.0,2208250.0
max,4.0,2840.0,24.0,1.0,5193000.0


SEPARATE FEATURES X & Y

In [ ]:
# Drop rows with missing size for now

data = df.dropna(subset=['size_sqft']).copy()

# X independent =  input the model reads.  We drop 'area' (its text, models need numbers - Step 5)

X = data[['bedrooms', 'size_sqft', 'age_years', 'freehold']]

# y dependent = Need to predict 

y = data['price_aed']

display(X.shape, y.shape, X.head()) 


(482, 4)

(482,)

,bedrooms,size_sqft,age_years,freehold
0,0,562.0,13,1
1,3,1961.0,8,0
2,4,2670.0,13,1
3,4,2593.0,4,0
4,1,630.0,2,1


In [7]:
X.info()

<class 'pandas.DataFrame'>
Index: 482 entries, 0 to 499
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   bedrooms   482 non-null    int64  
 1   size_sqft  482 non-null    float64
 2   age_years  482 non-null    int64  
 3   freehold   482 non-null    int64  
dtypes: float64(1), int64(3)
memory usage: 18.8 KB


SPLIT

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X,y,
                                                    test_size=0.2, # 20% 
                                                    random_state = 42 # makes the random split reproductible
)
display(len(X_train),len(X_test))


385

97

DUMB BASELINE 

Predict the avg, measure the error

In [9]:
from sklearn.metrics import mean_absolute_error

baseline_predictions = y_train.mean()
print("Baseline Price for every flat:", round(baseline_predictions))


# Make an array that predicts this same number for every test flat
baseline_preds = [baseline_predictions] * len(y_test)

# How wrong are we, on average, in AED?
baseline_mae = mean_absolute_error(y_test , baseline_preds)
print("Baseline MAE:", baseline_mae)

Baseline Price for every flat: 1537021
Baseline MAE: 914290.5877627525


Train and evaluate

In [10]:
from sklearn.linear_model import LinearRegression

# Create Model
model = LinearRegression()

# Train it ON TRAINING DATA ONLY -- this is "fit"
model.fit(X_train, y_train)

# Predict on the test data
pred = model.predict(X_test)

# Measure error --> MAE and compare with baseline
from sklearn.metrics import mean_absolute_error 
model_mae = mean_absolute_error(y_test , pred)

print("Baseline MAE",baseline_mae)
print("Model MAE", round(model_mae))
print("Improvement:",round(baseline_mae-model_mae))

Baseline MAE 914290.5877627525
Model MAE 398711
Improvement: 515580


One-hot encode and re-split

In [11]:
# Start Fresh from data, still dropping missing values

data = df.dropna(subset=['size_sqft']).copy()

# pd.get_dummies does one-hot encoding automatically
# It finds every unique area and makes a 0/1 column for each
data_encoded = pd.get_dummies(data, columns = ['area'])
print("Columns now:", list(data_encoded.columns))
data_encoded.head()

Columns now: ['bedrooms', 'size_sqft', 'age_years', 'freehold', 'price_aed', 'area_Al Barsha', 'area_Arabian Ranches', 'area_Business Bay', 'area_Deira', 'area_Downtown', 'area_Dubai Hills', 'area_Dubai Marina', 'area_JBR', 'area_JVC', 'area_Mirdif']


,bedrooms,size_sqft,age_years,freehold,price_aed,area_Al Barsha,area_Arabian Ranches,area_Business Bay,area_Deira,area_Downtown,area_Dubai Hills,area_Dubai Marina,area_JBR,area_JVC,area_Mirdif
0,0,562.0,13,1,664000,False,False,False,False,False,False,True,False,False,False
1,3,1961.0,8,0,1343000,False,False,False,False,False,False,False,False,False,True
2,4,2670.0,13,1,3413000,False,True,False,False,False,False,False,False,False,False
3,4,2593.0,4,0,2557000,False,False,False,False,False,False,False,False,True,False
4,1,630.0,2,1,618000,False,False,False,False,False,False,False,False,True,False


In [12]:
print(len(data_encoded.columns))

15


In [13]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import median_absolute_error
from sklearn.linear_model import LinearRegression

# y is still price. X is now EVERYTHING except price.

y_ = data_encoded['price_aed']
X_ = data_encoded.drop(columns=['price_aed'])

# SPLIT
X_train, X_test, y_train, y_test = train_test_split(X_,y_,test_size=0.2,random_state=42)

display(len(X_train),len(X_test))


385

97

In [14]:
model = LinearRegression()

model.fit(X_train,y_train)

predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
print("Baseline MAE:",round(baseline_mae))
print("Without area:",round(model_mae))
print("With area:", round(mae))


Baseline MAE: 914291
Without area: 398711
With area: 244992


size_per_bedroom = size_sqft / bedrooms — roominess (my example)

size_x_age = the interaction you were reaching for — does age hit big and small places differently
We'll keep freehold in and then inspect its weight to settle your prediction with evidence

In [15]:
data_fe = df.dropna(subset = ['size_sqft']).copy()

# Studios have 0 bedrooms -> treat them as 1 "room" to avoid divide-by-zero
rooms = data_fe['bedrooms'].replace(0,1)

# Size / Bedroom
data_fe['size_per_bedroom'] = round(data_fe['size_sqft'] / rooms,2)

# Size * Age
data_fe['size_x_age'] = data_fe['size_sqft'] * data_fe['age_years']

# data_fe = data_fe.assign(size_x_age = lambda x: x['size_sqft'] * x['age_years'])
# data_fe['size_x_age'] = data_fe.eval('size_sqft' * 'age_years')

data_fe = pd.get_dummies(data_fe, columns=['area'])
data_fe.head()



,bedrooms,size_sqft,age_years,freehold,price_aed,size_per_bedroom,size_x_age,area_Al Barsha,area_Arabian Ranches,area_Business Bay,area_Deira,area_Downtown,area_Dubai Hills,area_Dubai Marina,area_JBR,area_JVC,area_Mirdif
0,0,562.0,13,1,664000,562.00,7306.0,False,False,False,False,False,False,True,False,False,False
1,3,1961.0,8,0,1343000,653.67,15688.0,False,False,False,False,False,False,False,False,False,True
2,4,2670.0,13,1,3413000,667.50,34710.0,False,True,False,False,False,False,False,False,False,False
3,4,2593.0,4,0,2557000,648.25,10372.0,False,False,False,False,False,False,False,False,True,False
4,1,630.0,2,1,618000,630.00,1260.0,False,False,False,False,False,False,False,False,True,False


In [16]:
print(list(data_fe.columns))

['bedrooms', 'size_sqft', 'age_years', 'freehold', 'price_aed', 'size_per_bedroom', 'size_x_age', 'area_Al Barsha', 'area_Arabian Ranches', 'area_Business Bay', 'area_Deira', 'area_Downtown', 'area_Dubai Hills', 'area_Dubai Marina', 'area_JBR', 'area_JVC', 'area_Mirdif']


In [17]:
X = data_fe.drop(columns = ['price_aed'])
y = data_fe['price_aed']

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

predictions = model.predict(X_test)
new_mae = mean_absolute_error(y_test, predictions)

In [18]:
print(new_mae)
print(mae)

240600.86144452027
244992.11182308017


In [19]:
weights = pd.Series(model.coef_, index = X.columns).sort_values()

for name, value in weights.items():
    print(f"{name:25s} {value:>15,.0f}")

# pd.set_option('display.float_format', lambda v: f"{v:,.0f}")
# print(weights)


area_Deira                       -764,000
area_Mirdif                      -535,879
area_JVC                         -365,372
area_Al Barsha                   -352,206
age_years                            -701
size_x_age                            -16
size_per_bedroom                      175
size_sqft                           1,242
area_Arabian Ranches                1,285
bedrooms                           11,195
freehold                           68,385
area_Business Bay                  90,259
area_Dubai Hills                  231,125
area_Dubai Marina                 291,889
area_JBR                          590,311
area_Downtown                     812,590


#### RANDOM FOREST

In [35]:
from sklearn.ensemble import RandomForestRegressor

# new copy of data
data_rf = df.dropna(subset = ['size_sqft']).copy()
data_rf = pd.get_dummies(data_rf , columns = ['area'])

X = data_rf.drop(columns = ['price_aed'])
y = data_rf['price_aed']

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, random_state = 42)

model = RandomForestRegressor(n_estimators = 200 , random_state = 42)

model.fit(X_train,y_train)

predictions = model.predict(X_test)

rf_mae = mean_absolute_error(y_test,predictions)
print(rf_mae)


137448.91408934706
